## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer 
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, precision_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

## Loading Dataset

In [11]:
df = pd.read_csv('spam.csv', encoding='latin-1')

## Preprocessing

In [12]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [13]:
df.drop(columns =['Unnamed: 2','Unnamed: 3', 'Unnamed: 4', 'v1'], inplace =True)

In [14]:
df.head()

,v2
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."


In [15]:
df.rename(columns = {'v2': 'text'}, inplace=True)

In [16]:
df.head()

,text
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
df.isnull().sum()

text    0
dtype: int64

In [18]:
df.duplicated().sum()

403

In [19]:
df = df.drop_duplicates(keep='first')

In [20]:
df.duplicated().sum()

0

## Text Preprocessing

Preprocessing:

Text Cleaning:

Remove punctuation, numbers, and special characters.

Convert the text to lowercase.

Remove stop words (commonly used words like "the", "and", etc.).

Tokenize the messages (break them into words).

Apply lemmatization or stemming to reduce words to their root form.

In [21]:
ps = PorterStemmer()

In [22]:
def transform_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    
    y = []
    
    words = text.split()
    for i in words:
        if i not in stopwords.words('english'):
            y.append(ps.stem(i))
            
    return ' '.join(y)

In [23]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
transform_text(df['text'][0])

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [25]:
re.sub(r'[^\w\s]', '', "How! are you?")

'How are you'

In [26]:
df['text'] = df['text'].apply(transform_text)

In [27]:
df['text']

0       go jurong point crazi avail bugi n great world...
1                                   ok lar joke wif u oni
2       free entri 2 wkli comp win fa cup final tkt 21...
3                     u dun say earli hor u c alreadi say
4               nah dont think goe usf live around though
                              ...                        
5567    2nd time tri 2 contact u u å750 pound prize 2 ...
5568                             ì_ b go esplanad fr home
5569                              piti mood soani suggest
5570    guy bitch act like id interest buy someth els ...
5571                                       rofl true name
Name: text, Length: 5169, dtype: object

In [28]:
cv = CountVectorizer()
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)

In [29]:
x = cv.fit_transform(df['text'])

In [30]:
x

<5169x8069 sparse matrix of type '<class 'numpy.int64'>'
	with 43060 stored elements in Compressed Sparse Row format>

In [31]:
tfidf_res = tfidf.fit_transform(df['text'])

In [32]:
tfidf_res

<5169x3000 sparse matrix of type '<class 'numpy.float64'>'
	with 41401 stored elements in Compressed Sparse Row format>

In [90]:
# using clusyering Algo

kmeans = KMeans(n_clusters = 2, random_state = 42)
kmeansy = KMeans(n_clusters = 2, random_state = 42)


In [91]:
kmeans
kmeansy

KMeans(n_clusters=2, random_state=42)

In [106]:
kmeans.fit(x)
kmeansy.fit(tfidf_res)

C:\Users\PC\anaconda3\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


KMeans(n_clusters=2, random_state=42)

In [107]:
df['clusters_x'] = kmeans.labels_
df['clusters_y'] = kmeansy.labels_

In [108]:
df['clusters_x'].value_counts()

clusters_x
1    4905
0     264
Name: count, dtype: int64

In [109]:
df['clusters_y'].value_counts()

clusters_y
0    4701
1     468
Name: count, dtype: int64

In [110]:
print(df[df['clusters_y'] == 0]['text'].head())
print(df[df['clusters_y'] == 1]['text'].head())

0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri 2 wkli comp win fa cup final tkt 21...
3                  u dun say earli hor u c alreadi say
4            nah dont think goe usf live around though
Name: text, dtype: object
8     winner valu network custom select receivea å90...
9     mobil 11 month u r entitl updat latest colour ...
12    urgent 1 week free membership å100000 prize ja...
42    07732584351 rodger burn msg tri call repli sm ...
45                               callsmessagesmiss call
Name: text, dtype: object


In [111]:
# Assign labels based on the cluster interpretation
df['label'] = df['clusters_y'].apply(lambda x: 1 if x == 1 else 0)  # Assume cluster 1 is spam


In [112]:
df.sample(5)

,text,clusters_x,clusters_y,label
3872,happi sad one thing past good morn,1,0,0
3867,know peopl still town,1,0,0
684,want ask ì_ wait 4 finish lect co lect finish ...,1,0,0
2495,winner valu network custom hvae select receiv ...,1,1,1
4156,singl singl answer fight plu said broke didnt ...,1,0,0


In [125]:
# X and 

X = tfidf_res
Y = df['label']

In [126]:
X = X.toarray()

In [127]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [128]:
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size = 0.2, random_state = 42)

In [129]:
x_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [130]:
mnb = MultinomialNB()

In [131]:
mnb.fit(x_train, y_train)

MultinomialNB()

In [134]:
# Step 4: Predict and Evaluate
y_pred = mnb.predict(x_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Precision Score: ", precision_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 95.45%
Precision Score:  0.9305555555555556
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       925
           1       0.93      0.61      0.74       109

    accuracy                           0.95      1034
   macro avg       0.94      0.80      0.86      1034
weighted avg       0.95      0.95      0.95      1034



In [138]:
# Step 3: Train Random Forest Model
rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)


RandomForestClassifier(random_state=42)

In [139]:
# Step 4: Predict and Evaluate
y_pred1 = rf.predict(x_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred1) * 100:.2f}%")
print("Precision Score: ", precision_score(y_test, y_pred1))
print(classification_report(y_test, y_pred1))

Accuracy: 99.32%
Precision Score:  0.9811320754716981
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       925
           1       0.98      0.95      0.97       109

    accuracy                           0.99      1034
   macro avg       0.99      0.98      0.98      1034
weighted avg       0.99      0.99      0.99      1034



In [141]:
# Assuming `model` is your trained Random Forest or Naive Bayes model
# and `tfidf` is the TF-IDF Vectorizer used for training

# Sample messages
sample_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim your prize!",
    "Get rich quick! Exclusive offer for free access to premium content. Act now!",
    "Urgent: Your account has been compromised. Please click the link to secure your account immediately.",
    "Limited time offer: Buy one get one free on all products. Call now to claim your discount!",
    "You've been selected for a special promotion. Reply with your details to receive your free gift!",
    "Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.",
    "Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.",
    "Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.",
    "Reminder: Your dentist appointment is scheduled for 2 PM tomorrow. Please let us know if you need to reschedule.",
    "Hi, just a quick note to confirm our lunch meeting tomorrow at noon. Looking forward to it!"
]

# Vectorize the sample messages
sample_messages_transformed = tfidf.transform(sample_messages)

# Predict using the model
predictions = rf.predict(sample_messages_transformed)

# Output the predictions
for message, prediction in zip(sample_messages, predictions):
    label = 'Spam' if prediction == 1 else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")


Message: Congratulations! You've won a $1000 gift card. Click here to claim your prize!
Prediction: Spam

Message: Get rich quick! Exclusive offer for free access to premium content. Act now!
Prediction: Not Spam

Message: Urgent: Your account has been compromised. Please click the link to secure your account immediately.
Prediction: Not Spam

Message: Limited time offer: Buy one get one free on all products. Call now to claim your discount!
Prediction: Spam

Message: You've been selected for a special promotion. Reply with your details to receive your free gift!
Prediction: Not Spam

Message: Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.
Prediction: Not Spam

Message: Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.
Prediction: Not Spam

Message: Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.
Prediction: Not Spam

Message: Reminder: Your

In [144]:
# Load and prepare data
df = pd.read_csv('spam.csv', encoding='latin-1')
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
df.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)

# Preprocessing
def transform_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    return ' '.join(ps.stem(word) for word in words if word not in stop_words)

df['text'] = df['text'].apply(transform_text)

# Feature Extraction
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
X = tfidf.fit_transform(df['text'])
y = df['label'].apply(lambda x: 1 if x == 'spam' else 0)

# Clustering (for demonstration)
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X)
df['clusters'] = clusters

# Model Training and Evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Naive Bayes Model")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest Model")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# Testing with Sample Messages
def predict_message(message, model):
    transformed_message = tfidf.transform([transform_text(message)])
    return model.predict(transformed_message)[0]

sample_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim your prize!",
    "Get rich quick! Exclusive offer for free access to premium content. Act now!",
    "Urgent: Your account has been compromised. Please click the link to secure your account immediately.",
    "Limited time offer: Buy one get one free on all products. Call now to claim your discount!",
    "You've been selected for a special promotion. Reply with your details to receive your free gift!",
    "Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.",
    "Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.",
    "Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.",
    "Reminder: Your dentist appointment is scheduled for 2 PM tomorrow. Please let us know if you need to reschedule.",
    "Hi, just a quick note to confirm our lunch meeting tomorrow at noon. Looking forward to it!"
]

print("Testing with Naive Bayes Model:")
for message in sample_messages:
    prediction = predict_message(message, nb)
    label = 'Spam' if prediction == 1 else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")

print("Testing with Random Forest Model:")
for message in sample_messages:
    prediction = predict_message(message, rf)
    label = 'Spam' if prediction == 1 else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\PC\anaconda3\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


Naive Bayes Model
Accuracy: 0.9748803827751196
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      1453
           1       1.00      0.81      0.89       219

    accuracy                           0.97      1672
   macro avg       0.99      0.90      0.94      1672
weighted avg       0.98      0.97      0.97      1672

Random Forest Model
Accuracy: 0.9772727272727273
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      1453
           1       1.00      0.83      0.91       219

    accuracy                           0.98      1672
   macro avg       0.99      0.91      0.95      1672
weighted avg       0.98      0.98      0.98      1672

Testing with Naive Bayes Model:
Message: Congratulations! You've won a $1000 gift card. Click here to claim your prize!
Prediction: Spam

Message: Get rich quick! Exclusive offer for free access to prem

In [147]:
# Load and prepare data
df = pd.read_csv('spam.csv', encoding='latin-1')
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
df.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)

# Preprocessing function
def transform_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    return ' '.join(ps.stem(word) for word in words if word not in stop_words)

df['text'] = df['text'].apply(transform_text)

# Feature Extraction
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)

# Features and target variable
X = tfidf.fit_transform(df['text'])
y = df['label'].apply(lambda x: 1 if x == 'spam' else 0)

# Handling class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

# Naive Bayes Model with Grid Search
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Random Forest Model with Grid Search
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Naive Bayes Model")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Classification Report:\n", classification_report(y_test, y_pred_nb))

print("Random Forest Model")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Classification Report:\n", classification_report(y_test, y_pred_rf))

# Testing with Sample Messages
def predict_message(message, model):
    transformed_message = tfidf.transform([transform_text(message)])
    return model.predict(transformed_message)[0]

sample_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim your prize!",
    "Get rich quick! Exclusive offer for free access to premium content. Act now!",
    "Urgent: Your account has been compromised. Please click the link to secure your account immediately.",
    "Limited time offer: Buy one get one free on all products. Call now to claim your discount!",
    "You've been selected for a special promotion. Reply with your details to receive your free gift!",
    "Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.",
    "Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.",
    "Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.",
    "Reminder: Your dentist appointment is scheduled for 2 PM tomorrow. Please let us know if you need to reschedule.",
    "Hi, just a quick note to confirm our lunch meeting tomorrow at noon. Looking forward to it!"
]

print("Testing with Naive Bayes Model:")
for message in sample_messages:
    prediction = predict_message(message, nb)
    label = 'Spam' if prediction == 1 else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")

print("Testing with Random Forest Model:")
for message in sample_messages:
    prediction = predict_message(message, rf)
    label = 'Spam' if prediction == 1 else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")


Naive Bayes Model
Accuracy: 0.9595854922279793
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96      1454
           1       0.97      0.95      0.96      1441

    accuracy                           0.96      2895
   macro avg       0.96      0.96      0.96      2895
weighted avg       0.96      0.96      0.96      2895

Random Forest Model
Accuracy: 0.9906735751295337
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      1454
           1       0.99      0.99      0.99      1441

    accuracy                           0.99      2895
   macro avg       0.99      0.99      0.99      2895
weighted avg       0.99      0.99      0.99      2895

Testing with Naive Bayes Model:
Message: Congratulations! You've won a $1000 gift card. Click here to claim your prize!
Prediction: Spam

Message: Get rich quick! Exclusive offer for free access to prem

In [150]:
# Load data
df = pd.read_csv('spam.csv', encoding='latin-1')

# Drop unnecessary columns
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)
df.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)

# Preprocessing function
def transform_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    return ' '.join(ps.stem(word) for word in words if word not in stop_words)

df['text'] = df['text'].apply(transform_text)


# Convert text to TF-IDF features
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000)
X = tfidf.fit_transform(df['text'])


# Apply K-Means Clustering
num_clusters = 2  # You can adjust this based on your analysis
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

# Examine clusters
print("Cluster 0 samples:")
print(df[df['cluster'] == 0]['text'].head())

print("Cluster 1 samples:")
print(df[df['cluster'] == 1]['text'].head())



# Assign labels based on manual inspection
df['predicted_label'] = df['cluster'].apply(lambda x: 'spam' if x == 1 else 'not spam')

# Check the distribution
print(df['predicted_label'].value_counts())


# Create feature and target arrays
X_supervised = tfidf.transform(df['text'])  # Use the same TF-IDF vectorizer
y_supervised = df['predicted_label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_supervised, y_supervised, test_size=0.3, random_state=42)


# Train Naive Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)

# Evaluate Naive Bayes model
print("Naive Bayes Model")
print("Accuracy:", accuracy_score(y_test, nb_predictions))
print("Classification Report:\n", classification_report(y_test, nb_predictions))

# Train Random Forest model
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# Evaluate Random Forest model
print("Random Forest Model")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("Classification Report:\n", classification_report(y_test, rf_predictions))








C:\Users\PC\anaconda3\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


Cluster 0 samples:
0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri 2 wkli comp win fa cup final tkt 21...
3                  u dun say earli hor u c alreadi say
4            nah dont think goe usf live around though
Name: text, dtype: object
Cluster 1 samples:
8     winner valu network custom select receivea å90...
9     mobil 11 month u r entitl updat latest colour ...
12    urgent 1 week free membership å100000 prize ja...
42    07732584351 rodger burn msg tri call repli sm ...
45                               callsmessagesmiss call
Name: text, dtype: object
predicted_label
not spam    5048
spam         524
Name: count, dtype: int64
Naive Bayes Model
Accuracy: 0.9527511961722488
Classification Report:
               precision    recall  f1-score   support

    not spam       0.96      0.99      0.97      1499
        spam       0.92      0.60      0.72       173

    accuracy                           0.95 

In [151]:
# Testing with Sample Messages
def predict_message(message, model):
    transformed_message = tfidf.transform([transform_text(message)])
    return model.predict(transformed_message)[0]

sample_messages = [
    "Congratulations! You've won a $1000 gift card. Click here to claim your prize!",
    "Get rich quick! Exclusive offer for free access to premium content. Act now!",
    "Urgent: Your account has been compromised. Please click the link to secure your account immediately.",
    "Limited time offer: Buy one get one free on all products. Call now to claim your discount!",
    "You've been selected for a special promotion. Reply with your details to receive your free gift!",
    "Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.",
    "Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.",
    "Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.",
    "Reminder: Your dentist appointment is scheduled for 2 PM tomorrow. Please let us know if you need to reschedule.",
    "Hi, just a quick note to confirm our lunch meeting tomorrow at noon. Looking forward to it!"
]

print("Testing with Naive Bayes Model:")
for message in sample_messages:
    prediction = predict_message(message, nb_model)
    label = 'Spam' if prediction == 'spam' else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")

print("Testing with Random Forest Model:")
for message in sample_messages:
    prediction = predict_message(message, rf_model)
    label = 'Spam' if prediction == 'spam' else 'Not Spam'
    print(f"Message: {message}\nPrediction: {label}\n")

Testing with Naive Bayes Model:
Message: Congratulations! You've won a $1000 gift card. Click here to claim your prize!
Prediction: Not Spam

Message: Get rich quick! Exclusive offer for free access to premium content. Act now!
Prediction: Not Spam

Message: Urgent: Your account has been compromised. Please click the link to secure your account immediately.
Prediction: Not Spam

Message: Limited time offer: Buy one get one free on all products. Call now to claim your discount!
Prediction: Not Spam

Message: You've been selected for a special promotion. Reply with your details to receive your free gift!
Prediction: Not Spam

Message: Hi John, I hope you're having a great day. Just checking in to see if you received my previous email.
Prediction: Not Spam

Message: Meeting agenda for tomorrow's team discussion: 1. Project updates 2. Budget review 3. Q&A.
Prediction: Not Spam

Message: Thank you for your purchase. Your order has been shipped and will arrive within 5-7 business days.
Predi